In [ ]:
# ═══════════════════════════════════════════════════════════════
# COLAB SETUP
# ═══════════════════════════════════════════════════════════════

# 1. Install dependencies
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "torch", "datasets", "scikit-learn", "matplotlib"], check=True)

# 2. Verify GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Training will be slow. Go to Runtime > Change runtime type > T4 GPU")

# 3. Create all required directories
import os
os.makedirs("outputs", exist_ok=True)
os.makedirs("model", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("eval_results", exist_ok=True)
print("Directory structure ready.")

In [ ]:
import os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt
from collections import defaultdict

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
import torch
from sklearn.manifold import TSNE

# ── Import data from the data/ package ──────────────────────────────────────
from data.categories import CATEGORIES
from data.domains import DOMAINS
from data.manual_corrections import MANUAL_CORRECTIONS

# ── Create all required output directories ─────────────────────────────────

print(f'Loaded {len(CATEGORIES)} categories')
print(f'Loaded {len(DOMAINS)} domains')
print(f'Loaded {len(MANUAL_CORRECTIONS)} manual corrections')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: ZERO-SHOT BASELINE
# Before any training, see how well the base model already does.
# This gives you a benchmark to know if training actually helped.
# ─────────────────────────────────────────────────────────────────────────────

def run_zero_shot(categories, domains, model_name="intfloat/e5-base-v2"):
    """
    Embed all domain descriptions and all categories.
    Assign each category to its nearest domain by cosine similarity.
    No training. Pure semantic matching.

    Why e5-base-v2?
    - Designed for asymmetric retrieval (long description vs short query)
    - Better than generic sentence-bert for this exact use case
    - Free, runs locally
    """
    print("Loading model...")
    model = SentenceTransformer(model_name)

    # e5 models need a prefix to distinguish query vs passage
    # Domain descriptions = passages (what we're searching through)
    # Categories = queries (what we're searching with)
    domain_names = list(domains.keys())
    domain_texts = [f"passage: {desc.strip()}" for desc in domains.values()]
    category_texts = [f"query: {cat}" for cat in categories]

    print("Embedding domains...")
    domain_embeddings = model.encode(domain_texts, normalize_embeddings=True, show_progress_bar=False)

    print("Embedding categories...")
    category_embeddings = model.encode(category_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=64)

    # Cosine similarity matrix: (num_categories x num_domains)
    similarity_matrix = category_embeddings @ domain_embeddings.T

    results = {}
    for i, category in enumerate(categories):
        scores = similarity_matrix[i]
        best_domain_idx = scores.argmax()
        best_domain = domain_names[best_domain_idx]
        best_score = float(scores[best_domain_idx])

        # Flag low confidence — these are your problem cases
        confidence = "HIGH" if best_score > 0.75 else "MEDIUM" if best_score > 0.60 else "LOW"

        results[category] = {
            "domain": best_domain,
            "score": round(best_score, 4),
            "confidence": confidence,
            "all_scores": {domain_names[j]: round(float(scores[j]), 4) for j in range(len(domain_names))}
        }

    return results

# Run it
baseline_results = run_zero_shot(CATEGORIES, DOMAINS)

# Print results grouped by domain
by_domain = defaultdict(list)
for cat, result in baseline_results.items():
    by_domain[result["domain"]].append((cat, result["score"], result["confidence"]))

print("\n" + "="*60)
print("ZERO-SHOT RESULTS")
print("="*60)
for domain, cats in sorted(by_domain.items()):
    print(f"\n{domain} ({len(cats)} categories):")
    for cat, score, conf in sorted(cats, key=lambda x: x[1], reverse=True):
        flag = "⚠️" if conf == "LOW" else "❓" if conf == "MEDIUM" else "✓"
        print(f"  {flag} {cat:<45} {score:.4f}")

# Save to outputs/ directory
with open("outputs/baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)
print("\nSaved to outputs/baseline_results.json")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: BUILD TRAINING EXAMPLES
#
# You review the LOW and MEDIUM confidence results from Phase 1.
# For each wrong assignment, you create a correction in
# data/manual_corrections.py — the ONLY manual work.
#
# For each correction, we create:
# - A POSITIVE pair: (category, correct_domain_description) → should be similar
# - A NEGATIVE pair: (category, wrong_domain_description)  → should NOT be similar
#
# The model learns: pull correct pairs together, push wrong pairs apart.
# This is contrastive learning.
# ─────────────────────────────────────────────────────────────────────────────

def get_problem_cases(results, threshold=0.75):
    """
    Extract categories the model was uncertain about.
    These are the only ones you need to manually review.
    """
    problems = {cat: r for cat, r in results.items() if r["score"] < threshold}
    print(f"\nProblem cases (score < {threshold}): {len(problems)} out of {len(results)}")
    return problems

problems = get_problem_cases(baseline_results)

def build_training_examples(corrections, domains, categories, high_confidence_results):
    """
    Build InputExample objects for sentence-transformers training.

    We build TWO types of training signal:

    1. From your manual corrections (the hard cases)
    2. From high-confidence zero-shot results (free signal — model already got these right)
       We use these as anchors so training doesn't break what already works.
    """
    examples = []

    # --- Signal from manual corrections ---
    for category, correct_domain, wrong_domain in corrections:
        correct_desc = domains[correct_domain].strip()
        wrong_desc = domains[wrong_domain].strip()

        # Positive pair: score = 1.0 (should be identical in embedding space)
        examples.append(InputExample(
            texts=[category, correct_desc],
            label=1.0
        ))

        # Negative pair: score = 0.0 (should be far apart)
        examples.append(InputExample(
            texts=[category, wrong_desc],
            label=0.0
        ))

        # Also add category against all other wrong domains at 0.1
        # (slightly wrong but not the biggest offender)
        for domain_name, domain_desc in domains.items():
            if domain_name != correct_domain and domain_name != wrong_domain:
                examples.append(InputExample(
                    texts=[category, domain_desc.strip()],
                    label=0.1
                ))

    # --- Signal from high-confidence zero-shot results (free!) ---
    # For categories the model already gets right with high confidence,
    # add them as positive pairs so training doesn't break them
    for category, result in high_confidence_results.items():
        if result["confidence"] == "HIGH":
            correct_domain = result["domain"]
            correct_desc = domains[correct_domain].strip()
            examples.append(InputExample(
                texts=[category, correct_desc],
                label=1.0
            ))

    print(f"Total training examples: {len(examples)}")
    return examples

high_conf = {cat: r for cat, r in baseline_results.items() if r["confidence"] == "HIGH"}
training_examples = build_training_examples(MANUAL_CORRECTIONS, DOMAINS, CATEGORIES, high_conf)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: FINE-TUNE THE MODEL
#
# We use CosineSimilarityLoss:
# - Takes (text_a, text_b, score) triplets
# - Adjusts weights so cosine similarity between text_a and text_b
#   matches the target score (1.0 = identical, 0.0 = opposite)
#
# We're NOT training from scratch.
# We're nudging e5-base-v2's geometry to match your taxonomy.
# The base model keeps all its language understanding.
# We just reshape where your 11 domains sit in the space.
# ─────────────────────────────────────────────────────────────────────────────

def fine_tune(training_examples, base_model="intfloat/e5-base-v2", output_path="./model"):
    model = SentenceTransformer(base_model)

    train_dataloader = DataLoader(
        training_examples,
        shuffle=True,
        batch_size=16      # Reduce to 8 if you get OOM errors on CPU
    )

    train_loss = losses.CosineSimilarityLoss(model)

    print("\nStarting fine-tuning...")
    print(f"Training examples: {len(training_examples)}")
    print(f"Batch size: 16")
    print(f"Epochs: 4")
    print(f"Estimated time: ~5-15 min on CPU, ~2 min on GPU\n")

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=4,
        warmup_steps=10,           # Gradually ramp up learning rate at start
        output_path=output_path,   # Saves the model here after training
        show_progress_bar=True,
        checkpoint_save_steps=500,
    )

    print(f"\nModel saved to: {output_path}")
    return model

trained_model = fine_tune(training_examples)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4: RUN FINAL CLASSIFICATION WITH TRAINED MODEL
#
# Same logic as Phase 1 (zero-shot), but now using your fine-tuned model.
# Compare results to baseline to see what changed.
# ─────────────────────────────────────────────────────────────────────────────

def classify_all(categories, domains, model):
    domain_names = list(domains.keys())
    domain_texts = [f"passage: {desc.strip()}" for desc in domains.values()]
    category_texts = [f"query: {cat}" for cat in categories]

    domain_embeddings = model.encode(domain_texts, normalize_embeddings=True)
    category_embeddings = model.encode(category_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

    similarity_matrix = category_embeddings @ domain_embeddings.T

    results = {}
    for i, category in enumerate(categories):
        scores = similarity_matrix[i]
        best_idx = scores.argmax()
        results[category] = {
            "domain": domain_names[best_idx],
            "score": round(float(scores[best_idx]), 4),
            "confidence": "HIGH" if scores[best_idx] > 0.75 else "MEDIUM" if scores[best_idx] > 0.60 else "LOW"
        }
    return results

final_results = classify_all(CATEGORIES, DOMAINS, trained_model)

# Save final mapping to outputs/
with open("outputs/final_mapping.json", "w") as f:
    json.dump(final_results, f, indent=2)
print("Saved to outputs/final_mapping.json")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5: COMPARE BASELINE vs FINE-TUNED
# See exactly what changed. Any remaining LOW confidence = needs more corrections.
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("CHANGES AFTER FINE-TUNING")
print("="*60)

changes = []
for cat in CATEGORIES:
    before = baseline_results[cat]["domain"]
    after = final_results[cat]["domain"]
    if before != after:
        change = {
            "category": cat,
            "before_domain": before,
            "before_score": baseline_results[cat]["score"],
            "after_domain": after,
            "after_score": final_results[cat]["score"],
        }
        changes.append(change)
        print(f"  {cat}")
        print(f"    BEFORE: {before} ({baseline_results[cat]['score']:.4f})")
        print(f"    AFTER:  {after}  ({final_results[cat]['score']:.4f})")

print(f"\nTotal changed: {len(changes)} categories")

# ── Confidence breakdown ─────────────────────────────────────────────────
conf_counts = defaultdict(int)
for cat, r in final_results.items():
    conf_counts[r['confidence']] += 1
print(f"\nConfidence breakdown (final):")
for level in ['HIGH', 'MEDIUM', 'LOW']:
    print(f"  {level}: {conf_counts[level]}")

# ── Save changes report ──────────────────────────────────────────────────
changes_report = {
    "total_categories": len(CATEGORIES),
    "total_changed": len(changes),
    "confidence_breakdown": dict(conf_counts),
    "changes": changes,
}
with open("outputs/changes_report.json", "w") as f:
    json.dump(changes_report, f, indent=2)

print("\nChanges report saved to: outputs/changes_report.json")
print("Final lookup table saved to: outputs/final_mapping.json")
print("Use this as a static dictionary for all future lookups.")
print("No model needed at runtime — just dict[category] → domain")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 6: PRODUCTION USAGE
# For your 2000 categories this is just a dictionary lookup.
# For NEW unseen categories, use the trained model.
# ─────────────────────────────────────────────────────────────────────────────

# Load the saved lookup table
with open("outputs/final_mapping.json") as f:
    LOOKUP_TABLE = json.load(f)

# Load the trained model for new unseen categories
production_model = SentenceTransformer("./model")
domain_names = list(DOMAINS.keys())
domain_embeddings = production_model.encode(
    [f"passage: {desc.strip()}" for desc in DOMAINS.values()],
    normalize_embeddings=True
)

def classify(category: str) -> dict:
    """
    Classify any category string into a domain.
    - Known categories: instant dictionary lookup (microseconds)
    - Unknown categories: model inference (milliseconds)
    """
    if category in LOOKUP_TABLE:
        return LOOKUP_TABLE[category]  # Instant

    # Unseen category — use trained model
    embedding = production_model.encode(
        f"query: {category}",
        normalize_embeddings=True
    )
    scores = embedding @ domain_embeddings.T
    best_idx = scores.argmax()
    return {
        "domain": domain_names[best_idx],
        "score": round(float(scores[best_idx]), 4),
        "confidence": "HIGH" if scores[best_idx] > 0.75 else "MEDIUM" if scores[best_idx] > 0.60 else "LOW"
    }

# Test it
print(classify("autonomous vehicle charging depot"))   # Never seen — uses model
print(classify("power station"))                       # Known — dict lookup

# ═══════════════════════════════════════════════════════════════
# Figures for Research Paper
# ═══════════════════════════════════════════════════════════════

All figures saved at **300 DPI** to `figures/` directory.

In [ ]:
def plot_confidence_distribution(baseline_results, final_results):
    """
    Figure 1: Histogram comparing cosine similarity score distributions
    before and after fine-tuning.
    """
    baseline_scores = [r['score'] for r in baseline_results.values()]
    finetuned_scores = [r['score'] for r in final_results.values()]

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.hist(baseline_scores, bins=50, alpha=0.6, color='#e74c3c', label='Baseline (zero-shot)', edgecolor='white', linewidth=0.5)
    ax.hist(finetuned_scores, bins=50, alpha=0.6, color='#2ecc71', label='Fine-tuned', edgecolor='white', linewidth=0.5)

    # Threshold lines
    ax.axvline(x=0.75, color='#f39c12', linestyle='--', linewidth=2, label='HIGH threshold (0.75)')
    ax.axvline(x=0.60, color='#9b59b6', linestyle='--', linewidth=2, label='MEDIUM threshold (0.60)')

    ax.set_xlabel('Cosine Similarity Score', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Categories', fontsize=13, fontweight='bold')
    ax.set_title('Confidence Distribution: Baseline vs Fine-Tuned', fontsize=15, fontweight='bold', pad=15)
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xlim(0.3, 1.0)

    plt.tight_layout()
    plt.savefig('figures/confidence_dist.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: figures/confidence_dist.png')

def plot_per_domain_accuracy(baseline_results, final_results, manual_corrections):
    """
    Figure 2: Grouped bar chart showing per-domain accuracy
    before and after fine-tuning, based on ground truth from MANUAL_CORRECTIONS.
    """
    # Build ground truth from manual corrections
    ground_truth = {cat: correct for cat, correct, wrong in manual_corrections}

    if len(ground_truth) == 0:
        print('WARNING: MANUAL_CORRECTIONS is empty — skipping per-domain accuracy plot.')
        print('Add corrections to data/manual_corrections.py and re-run.')
        return

    # Get all domains that appear in ground truth
    gt_domains = sorted(set(ground_truth.values()))

    # Calculate accuracy per domain
    baseline_correct = defaultdict(int)
    finetuned_correct = defaultdict(int)
    domain_total = defaultdict(int)

    for cat, true_domain in ground_truth.items():
        domain_total[true_domain] += 1
        if cat in baseline_results and baseline_results[cat]['domain'] == true_domain:
            baseline_correct[true_domain] += 1
        if cat in final_results and final_results[cat]['domain'] == true_domain:
            finetuned_correct[true_domain] += 1

    # Build bar data
    domains_list = gt_domains
    baseline_acc = [100.0 * baseline_correct[d] / domain_total[d] if domain_total[d] > 0 else 0 for d in domains_list]
    finetuned_acc = [100.0 * finetuned_correct[d] / domain_total[d] if domain_total[d] > 0 else 0 for d in domains_list]

    x = np.arange(len(domains_list))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 7))
    bars1 = ax.bar(x - width/2, baseline_acc, width, label='Baseline', color='#e74c3c', alpha=0.85, edgecolor='white')
    bars2 = ax.bar(x + width/2, finetuned_acc, width, label='Fine-tuned', color='#2ecc71', alpha=0.85, edgecolor='white')

    # Value labels on bars
    for bar in bars1:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Short domain labels
    short_labels = [d.split('_', 1)[1] if '_' in d else d for d in domains_list]
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=35, ha='right', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
    ax.set_title('Per-Domain Classification Accuracy: Baseline vs Fine-Tuned', fontsize=15, fontweight='bold', pad=15)
    ax.set_ylim(0, 115)
    ax.legend(fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('figures/per_domain_accuracy.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: figures/per_domain_accuracy.png')

def plot_tsne(categories, domains, model, title, save_path):
    """
    Figure 3: t-SNE scatter plot of all category embeddings,
    colored by assigned domain. Domain anchors plotted as stars.
    """
    domain_names = list(domains.keys())
    domain_texts = [f'passage: {desc.strip()}' for desc in domains.values()]
    category_texts = [f'query: {cat}' for cat in categories]

    print(f'  Encoding {len(categories)} categories...')
    domain_embeddings = model.encode(domain_texts, normalize_embeddings=True, show_progress_bar=False)
    category_embeddings = model.encode(category_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

    # Assign each category to nearest domain
    sim_matrix = category_embeddings @ domain_embeddings.T
    assignments = [domain_names[row.argmax()] for row in sim_matrix]

    # Combine all embeddings for t-SNE
    all_embeddings = np.vstack([category_embeddings, domain_embeddings])

    print('  Running t-SNE (this may take a minute)...')
    tsne = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000)
    coords = tsne.fit_transform(all_embeddings)

    cat_coords = coords[:len(categories)]
    dom_coords = coords[len(categories):]

    # Color map for domains
    palette = [
        '#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6',
        '#1abc9c', '#e67e22', '#34495e', '#e91e63', '#00bcd4',
        '#8bc34a', '#795548',
    ]
    domain_to_color = {name: palette[i % len(palette)] for i, name in enumerate(domain_names)}

    fig, ax = plt.subplots(figsize=(16, 12))

    # Plot category points
    for domain in domain_names:
        mask = [i for i, a in enumerate(assignments) if a == domain]
        if mask:
            short = domain.split('_', 1)[1] if '_' in domain else domain
            ax.scatter(
                cat_coords[mask, 0], cat_coords[mask, 1],
                c=domain_to_color[domain], s=12, alpha=0.5,
                label=f'{short} ({len(mask)})', edgecolors='none'
            )

    # Plot domain anchors as stars
    for j, name in enumerate(domain_names):
        short = name.split('_', 1)[1] if '_' in name else name
        ax.scatter(dom_coords[j, 0], dom_coords[j, 1],
                   marker='*', s=350, c=domain_to_color[name],
                   edgecolors='black', linewidths=1.2, zorder=5)
        ax.annotate(short, (dom_coords[j, 0], dom_coords[j, 1]),
                    fontsize=9, fontweight='bold',
                    xytext=(8, 8), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray'))

    ax.set_title(title, fontsize=15, fontweight='bold', pad=15)
    ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
    ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
    ax.legend(fontsize=9, loc='upper right', ncol=2, framealpha=0.9)
    ax.grid(alpha=0.15)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {save_path}')

def plot_suffix_analysis(categories, baseline_results):
    """
    Figure 4: Two side-by-side bar charts analyzing suffix poisoning.
    Left: count of categories per suffix. Right: mean cosine similarity per suffix.
    """
    suffixes = ['station', 'company', 'supplier', 'service', 'center', 'contractor']

    counts = []
    mean_scores = []

    for suffix in suffixes:
        matching = [cat for cat in categories if cat.endswith(suffix)]
        counts.append(len(matching))
        if matching:
            scores = [baseline_results[cat]['score'] for cat in matching if cat in baseline_results]
            mean_scores.append(np.mean(scores) if scores else 0.0)
        else:
            mean_scores.append(0.0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6', '#1abc9c']

    # Left chart: counts
    bars1 = ax1.bar(suffixes, counts, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    for bar, c in zip(bars1, counts):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(c), ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Suffix', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Number of Categories', fontsize=13, fontweight='bold')
    ax1.set_title('Category Count by Suffix', fontsize=14, fontweight='bold', pad=12)
    ax1.grid(axis='y', alpha=0.3)
    ax1.tick_params(axis='x', labelsize=11)

    # Right chart: mean cosine similarity
    bars2 = ax2.bar(suffixes, mean_scores, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    for bar, s in zip(bars2, mean_scores):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{s:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax2.axhline(y=0.75, color='#f39c12', linestyle='--', linewidth=1.5, label='HIGH threshold')
    ax2.axhline(y=0.60, color='#9b59b6', linestyle='--', linewidth=1.5, label='MEDIUM threshold')
    ax2.set_xlabel('Suffix', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Mean Cosine Similarity', fontsize=13, fontweight='bold')
    ax2.set_title('Mean Baseline Score by Suffix', fontsize=14, fontweight='bold', pad=12)
    ax2.set_ylim(0, 1.0)
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)
    ax2.tick_params(axis='x', labelsize=11)

    plt.suptitle('Suffix Poisoning Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('figures/suffix_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: figures/suffix_analysis.png')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Generate all figures
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 60)
print('GENERATING ALL FIGURES')
print('=' * 60)

# Figure 1
print('\n[1/5] Confidence Distribution...')
plot_confidence_distribution(baseline_results, final_results)

# Figure 2
print('\n[2/5] Per-Domain Accuracy...')
plot_per_domain_accuracy(baseline_results, final_results, MANUAL_CORRECTIONS)

# Figure 3a — t-SNE with base model
print('\n[3/5] t-SNE Baseline Embedding Space...')
base_model_for_tsne = SentenceTransformer('intfloat/e5-base-v2')
plot_tsne(CATEGORIES, DOMAINS, base_model_for_tsne,
          't-SNE: Baseline Embedding Space (e5-base-v2)', 'figures/tsne_baseline.png')
del base_model_for_tsne  # free memory

# Figure 3b — t-SNE with fine-tuned model
print('\n[4/5] t-SNE Fine-Tuned Embedding Space...')
ft_model_for_tsne = SentenceTransformer('./model')
plot_tsne(CATEGORIES, DOMAINS, ft_model_for_tsne,
          't-SNE: Fine-Tuned Embedding Space', 'figures/tsne_finetuned.png')
del ft_model_for_tsne  # free memory

# Figure 4
print('\n[5/5] Suffix Poisoning Analysis...')
plot_suffix_analysis(CATEGORIES, baseline_results)

print('\n' + '=' * 60)
print('ALL FIGURES SAVED TO figures/')
print('=' * 60)
